# Transformers Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration pour la reproductibilité
torch.manual_seed(42)

# Hyperparamètres inspirés du document
d_model = 512  # Dimension des embeddings (d)
n_head = 8     # Nombre de têtes d'attention (h)
d_k = d_model // n_head # Dimension par tête
max_len = 100  # Longueur max de la séquence

Cellule 2 : Encodage Positionnel (Positional Encoding)Théorie : Le modèle n'a pas de notion d'ordre. On injecte des sinusoïdes selon la formule:
$$PE(pos, 2i) = \sin(pos/10000^{2i/d_{model}})$$
$$PE(pos, 2i+1) = \cos(pos/10000^{2i/d_{model}})$$

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # Création de la matrice PE (Position, Dimension)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0)) # Shape: (1, max_len, d_model)

    def forward(self, x):
        # Ajout direct aux embeddings d'entrée [cite: 805]
        x = x + self.pe[:, :x.size(1)]
        return x

# Visualisation (similaire à la slide 34 [cite: 714])
pe_layer = PositionalEncoding(d_model=128, max_len=50)
pe_matrix = pe_layer.pe.squeeze().numpy()

plt.figure(figsize=(10, 6))
sns.heatmap(pe_matrix, cmap="RdBu", center=0)
plt.title("Visualisation des Embeddings Positionnels (Pos vs Depth)")
plt.xlabel("Profondeur de l'embedding (d)")
plt.ylabel("Position dans la séquence")
plt.show()

Cellule 3 : Scaled Dot-Product Attention (Mécanisme Unique)Théorie : C'est le cœur du mécanisme. On calcule la similarité entre Query et Key, on met à l'échelle (Scale), on applique Softmax, puis on pondère les Values.
$$Attention(Q, K, V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}})V$$

In [ ]:
def scaled_dot_product_attention(query, key, value, mask=None):
    """
    Calcule l'attention selon l'équation de la slide 19[cite: 422].
    Inputs:
        query, key, value: Tenseurs de dimension (Batch, Heads, Seq_Len, d_k)
    """
    d_k = query.size(-1)
    
    # 1. MatMul Q * K^T
    # Transpose des deux dernières dimensions pour la multiplication matricielle
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    
    # 2. Masking (Optionnel, utilisé dans le décodeur [cite: 392])
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e9)
    
    # 3. Softmax [cite: 243]
    p_attn = F.softmax(scores, dim=-1)
    
    # 4. MatMul avec V
    output = torch.matmul(p_attn, value)
    
    return output, p_attn

# --- Test Unitaire Mathématique ---
# Simulation de 3 mots avec dimension 4 (Slide 10 montre des vecteurs simples [cite: 162])
seq_len = 3
dim = 4
q = torch.randn(1, 1, seq_len, dim)
k = torch.randn(1, 1, seq_len, dim)
v = torch.randn(1, 1, seq_len, dim)

out, attn_weights = scaled_dot_product_attention(q, k, v)

print("Score d'attention (Matrice 3x3 normalized) :\n", attn_weights[0,0])
print("Somme des probabilités (doit être 1) :", attn_weights[0,0].sum(dim=1))

Cellule 4 : Multi-Head Attention (MHA)Théorie : Plutôt que de faire une seule attention sur tout le vecteur $d$, on projette en $h$ sous-espaces pour capturer différentes relations.
$$MultiHead(Q, K, V) = Concat(head_1, ..., head_h)W^O$$

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        assert d_model % n_head == 0, "d_model doit être divisible par n_head"
        
        self.d_k = d_model // n_head
        self.n_head = n_head
        
        # Matrices de projection Wq, Wk, Wv [cite: 333]
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model) # Wo [cite: 453]
        
    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        
        # 1. Projections linéaires et séparation en têtes
        # On reshape de (Batch, Seq, D) -> (Batch, Seq, Head, D_k) -> (Batch, Head, Seq, D_k)
        query = self.w_q(q).view(batch_size, -1, self.n_head, self.d_k).transpose(1, 2)
        key = self.w_k(k).view(batch_size, -1, self.n_head, self.d_k).transpose(1, 2)
        value = self.w_v(v).view(batch_size, -1, self.n_head, self.d_k).transpose(1, 2)
        
        # 2. Scaled Dot-Product Attention sur toutes les têtes en parallèle [cite: 438]
        out, self.attn_weights = scaled_dot_product_attention(query, key, value, mask)
        
        # 3. Concaténation [cite: 449]
        # (Batch, Head, Seq, D_k) -> (Batch, Seq, Head, D_k) -> (Batch, Seq, D_model)
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        
        # 4. Projection finale linéaire
        return self.fc_out(out)

# Note: Ajout d'attribut pour le constructeur
MultiHeadAttention.d_model = d_model

Cellule 5 : Feed Forward & Layer Norm (Le bloc complet)Théorie : Chaque sous-couche (Attention et FFN) est entourée d'une connexion résiduelle et d'une normalisation.
$$LayerNorm(x + Sublayer(x))$$
Le FFN est une transformation non-linéaire : $FFN(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$.

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048):
        super().__init__()
        # d_ff est généralement 4x plus grand que d_model [cite: 472]
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        
    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.mha = MultiHeadAttention(d_model, n_head)
        self.ffn = FeedForward(d_model, d_ff)
        
        # Normalisation de couche [cite: 488]
        self.layernorm1 = nn.LayerNorm(d_model)
        self.layernorm2 = nn.LayerNorm(d_model)
        
    def forward(self, x, mask=None):
        # Sous-couche 1 : MHA + Residual + Norm [cite: 498]
        attn_output = self.mha(x, x, x, mask) # Self-attention: Q=K=V=x
        x = self.layernorm1(x + attn_output)
        
        # Sous-couche 2 : FFN + Residual + Norm [cite: 499]
        ffn_output = self.ffn(x)
        x = self.layernorm2(x + ffn_output)
        
        return x

Cellule 6 : Simulation "Jouet" et Visualisation de l'Attention
Ici, nous simulons une phrase et regardons comment les mots s'influencent mutuellement (Self-Attention).

In [ ]:
# --- Création d'une "Phrase" Mathématique ---
# Supposons une séquence de 5 "mots" (vecteurs aléatoires)
seq_length = 5
input_tensor = torch.randn(1, seq_length, d_model)

# --- Passage dans le Bloc Encodeur ---
# Instanciation
encoder_block = EncoderLayer(d_model=d_model, n_head=n_head, d_ff=2048)

# Forward pass
output = encoder_block(input_tensor)

print(f"Input Shape: {input_tensor.shape}")
print(f"Output Shape: {output.shape} (Doit être identique à l'input)")

# --- Visualisation des poids d'attention ---
# Récupération des poids stockés dans l'instance MHA
# Shape: (Batch, Heads, Seq_Len, Seq_Len)
attn = encoder_block.mha.attn_weights.detach().squeeze(0).numpy()

# Plot des 8 têtes d'attention
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
tokens = ["Le", "chat", "est", "sur", "le", "tapis"][:seq_length] # Labels fictifs

for h in range(n_head):
    row = h // 4
    col = h % 4
    ax = axes[row, col]
    sns.heatmap(attn[h], annot=True, fmt=".2f", cmap="viridis", xticklabels=tokens, yticklabels=tokens, ax=ax, cbar=False)
    ax.set_title(f"Head {h+1}")

plt.suptitle("Attention Multi-Têtes : Qui regarde qui ? [cite: 423]", fontsize=16)
plt.show()

Cellule 7 : Le Masque "Look-Ahead" (Causalité)Théorie : Dans le décodeur, pour prédire le mot à la position $t$, le modèle ne doit pas avoir accès aux mots des positions $t+1$. On applique un masque triangulaire supérieur avec des $-\infty$ avant le Softmax.$$Mask_{i,j} = \begin{cases} 0 & \text{si } i \ge j \\ -\infty & \text{si } i < j \end{cases}$$

In [ ]:
def create_look_ahead_mask(size):
    """
    Crée un masque triangulaire supérieur pour empêcher
    le décodeur de regarder le futur.
    """
    mask = torch.triu(torch.ones(size, size), diagonal=1)
    return mask == 1  # Retourne un masque booléen (True = à masquer)

# Visualisation du masque
mask_size = 10
mask = create_look_ahead_mask(mask_size)

plt.figure(figsize=(6, 5))
sns.heatmap(mask, cmap="Reds", cbar=False, linewidths=0.5, linecolor='gray')
plt.title("Masque Causal (Look-Ahead)\nLes zones rouges sont interdites d'attention")
plt.xlabel("Key Position (Futur)")
plt.ylabel("Query Position (Présent)")
plt.show()

Cellule 8 : La Couche Décodeur (Cross-Attention)Théorie : Le décodeur possède une sous-couche supplémentaire par rapport à l'encodeur : l'Attention Encodeur-Décodeur (Cross-Attention).Masked Self-Attention : Le décodeur s'analyse lui-même (avec masque causal).Encoder-Decoder Attention :Query ($Q$) : Vient de la couche précédente du Décodeur.Key ($K$) & Value ($V$) : Viennent de la sortie finale de l'Encodeur.Cela permet au décodeur de se "focaliser" sur les parties pertinentes de la source (ex: phrase en français) pour générer la cible (ex: mot en anglais).

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        # 1. Self-Attention (Masquée)
        self.self_mha = MultiHeadAttention(d_model, n_head)
        
        # 2. Cross-Attention (Encoder-Decoder)
        self.cross_mha = MultiHeadAttention(d_model, n_head)
        
        # 3. Feed Forward
        self.ffn = FeedForward(d_model, d_ff)
        
        # Normalisations
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
    def forward(self, x, enc_output, look_ahead_mask, padding_mask):
        # A. Masked Self-Attention (x regarde x)
        # Le masque look_ahead est appliqué ici
        attn1, w1 = self.self_mha(x, x, x, mask=look_ahead_mask)
        x = self.norm1(x + attn1)
        
        # B. Encoder-Decoder Attention (x regarde enc_output)
        # Query = x (décodeur), Key/Value = enc_output (encodeur)
        # Le padding_mask sert à ignorer les tokens de padding de l'entrée source
        attn2, w2 = self.cross_mha(q=x, k=enc_output, v=enc_output, mask=padding_mask)
        x = self.norm2(x + attn2)
        
        # C. Feed Forward
        ffn_out = self.ffn(x)
        x = self.norm3(x + ffn_out)
        
        return x, w1, w2

Cellule 9 : Architecture Transformer Complète (Toy Model)
Assemblage des pièces : Embedding + Positional Encoding + N couches Encodeurs + N couches Décodeurs + Projection Finale.

In [ ]:
class Transformer(nn.Module):
    def __init__(self, d_model, n_head, d_ff, src_vocab_size, tgt_vocab_size, num_layers=2):
        super().__init__()
        
        # --- Composants Encodeur ---
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.enc_pe = PositionalEncoding(d_model)
        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, n_head, d_ff) for _ in range(num_layers)])
        
        # --- Composants Décodeur ---
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.dec_pe = PositionalEncoding(d_model)
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, n_head, d_ff) for _ in range(num_layers)])
        
        # --- Sortie ---
        self.final_linear = nn.Linear(d_model, tgt_vocab_size)
        
    def encode(self, src):
        x = self.encoder_embedding(src)
        x = self.enc_pe(x)
        for layer in self.encoder_layers:
            x = layer(x)
        return x # (Batch, Src_Len, d_model)
    
    def decode(self, tgt, enc_output, look_ahead_mask):
        x = self.decoder_embedding(tgt)
        x = self.dec_pe(x)
        attention_weights = {}
        
        for i, layer in enumerate(self.decoder_layers):
            x, w_self, w_cross = layer(x, enc_output, look_ahead_mask, padding_mask=None)
            attention_weights[f'dec_layer_{i}_cross'] = w_cross
            
        return x, attention_weights

    def forward(self, src, tgt):
        # 1. Encodage
        enc_output = self.encode(src)
        
        # 2. Masque pour le décodeur
        seq_len = tgt.size(1)
        look_ahead_mask = create_look_ahead_mask(seq_len).to(tgt.device)
        
        # 3. Décodage
        dec_output, weights = self.decode(tgt, enc_output, look_ahead_mask)
        
        # 4. Projection vers le vocabulaire
        logits = self.final_linear(dec_output)
        return logits, weights

Cellule 10 : Simulation d'Inférence (Génération Mot-à-Mot)Concept : Contrairement à l'entraînement (où l'on donne toute la phrase cible masquée d'un coup), en inférence, on boucle. On génère le token $t$, on l'ajoute à l'entrée, et on génère $t+1$. C'est le goulot d'étranglement mentionné slide 28.

In [ ]:
# Simulation de chargement d'un fichier texte
raw_text = """
Les transformeurs sont des modèles puissants.
L'attention est tout ce dont vous avez besoin.
Le cache KV accélère la génération.
"""

# Création d'un vocabulaire simple (niveau caractère pour simplifier, ou mot)
class SimpleTokenizer:
    def __init__(self, text):
        # On extrait les mots uniques
        self.words = sorted(list(set(text.replace('\n', ' ').split())))
        self.words = ["<pad>", "<start>", "<end>"] + self.words
        self.word2idx = {w: i for i, w in enumerate(self.words)}
        self.idx2word = {i: w for i, w in enumerate(self.words)}
        self.vocab_size = len(self.words)
    
    def encode(self, sentence):
        return [self.word2idx.get(w, 0) for w in sentence.split()]
    
    def decode(self, indices):
        return " ".join([self.idx2word[idx] for idx in indices])

tokenizer = SimpleTokenizer(raw_text)
print(f"Taille du vocabulaire : {tokenizer.vocab_size}")
print(f"Exemple encodage 'cache' : {tokenizer.word2idx['cache']}")

In [ ]:
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer

# --- 1. Création d'un fichier texte factice ---
text_content = "Le mécanisme d'attention est puissant."
with open("source_text.txt", "w", encoding="utf-8") as f:
    f.write(text_content)

# --- 2. Chargement d'un Tokenizer Classique (BERT) ---
# 'bert-base-multilingual-cased' est adapté car il gère le français et l'anglais
tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

# Lecture et tokenisation du fichier
with open("source_text.txt", "r", encoding="utf-8") as f:
    raw_text = f.read().strip()

# Encodage de la phrase source (transforme les mots en ID du vocabulaire BERT)
# return_tensors="pt" renvoie directement des tenseurs PyTorch
src_seq = tokenizer.encode(raw_text, return_tensors="pt")

print(f"Phrase source : '{raw_text}'")
print(f"Tokens source IDs : {src_seq}")
print(f"Tokens décodés pour vérification : {tokenizer.convert_ids_to_tokens(src_seq[0])}")

# --- 3. Configuration du Modèle ---
# On utilise la taille du vocabulaire réel du tokenizer (env. 119k pour mBERT)
vocab_size = tokenizer.vocab_size

# Instanciation du modèle (défini dans les cellules précédentes)
# Note : Le modèle est NON-ENTRAÎNÉ (poids aléatoires)
model = Transformer(d_model=512, n_head=8, d_ff=1024, 
                    src_vocab_size=vocab_size, tgt_vocab_size=vocab_size)

# --- 4. Préparation du Décodeur ---
# On commence avec le token spécial de début de phrase ([CLS] pour BERT)
start_token_id = tokenizer.cls_token_id
decoder_input = torch.tensor([[start_token_id]])

print("\n--- Démarrage de la génération (Modele non-entraîné = Sortie aléatoire) ---")

# --- 5. Boucle de Génération (Greedy Search) ---
max_new_tokens = 5
model.eval()

with torch.no_grad():
    # A. Encodage unique de la source
    enc_output = model.encode(src_seq)
    
    for i in range(max_new_tokens):
        # B. Masque causal pour le décodeur
        mask = create_look_ahead_mask(decoder_input.size(1))
        
        # C. Passage dans le décodeur
        dec_output, weights = model.decode(decoder_input, enc_output, mask)
        
        # D. Prédiction du prochain token (dernier vecteur de la séquence)
        last_token_logits = model.final_linear(dec_output[:, -1, :])
        predicted_id = torch.argmax(last_token_logits, dim=-1)
        
        # E. Concaténation
        decoder_input = torch.cat([decoder_input, predicted_id.unsqueeze(0)], dim=1)
        
        # Affichage du mot prédit (décodage de l'ID vers le string)
        predicted_word = tokenizer.decode(predicted_id)
        print(f"Étape {i+1}: Token ID {predicted_id.item()} -> Mot: '{predicted_word}'")

print(f"\nPhrase finale générée : {tokenizer.decode(decoder_input[0])}")

# --- 6. Visualisation de l'Attention Croisée ---
# Récupération des poids d'attention de la dernière étape
# weights['dec_layer_1_cross'] shape : (Batch, Heads, Tgt_Len, Src_Len)
# On regarde la première tête (0) et le dernier token généré (-1)
cross_attn = weights['dec_layer_0_cross'][0, 0, :, :].detach().numpy()

# Labels pour les axes (conversion des IDs en texte lisible)
src_labels = tokenizer.convert_ids_to_tokens(src_seq[0])
gen_labels = tokenizer.convert_ids_to_tokens(decoder_input[0])

plt.figure(figsize=(10, 8))
sns.heatmap(cross_attn, cmap="Blues", 
            xticklabels=src_labels,
            yticklabels=gen_labels)
plt.title("Cross-Attention Map : Quels mots source ont influencé la génération ?")
plt.xlabel("Source (Encoder)")
plt.ylabel("Génération (Decoder)")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.show()

In [ ]:
import torch.optim as optim

# --- 1. Préparation des Tenseurs (Input vs Target) ---
# Dans une tâche de reconstruction (Auto-Encoder), la source et la cible sont identiques.
# Source : [CLS] Le mécanisme ... [SEP]
# Decoder Input : [CLS] Le mécanisme ...
# Decoder Target (Labels) : Le mécanisme ... [SEP]

# On reprend la séquence tokenisée précédente
# src_seq contient déjà les IDs de "Le mécanisme d'attention est puissant." (avec CLS et SEP)
# Pour BERT : [CLS] = 101, [SEP] = 102

# Création des entrées/sorties pour le décodeur
# On enlève le dernier token ([SEP]) pour l'entrée du décodeur
decoder_input_train = src_seq[:, :-1] 

# On enlève le premier token ([CLS]) pour les labels (ce qu'on doit prédire)
# On veut que le modèle prédise le mot SUIVANT à chaque étape.
decoder_targets_train = src_seq[:, 1:]

print(f"Decoder Input shape: {decoder_input_train.shape}")
print(f"Decoder Target shape: {decoder_targets_train.shape}")

# --- 2. Configuration de l'Entraînement ---
# On remet le modèle en mode entraînement (active le Dropout si présent, etc.)
model.train()

# Optimiseur Adam (Standard pour Transformers)
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Fonction de perte (ignore l'index de padding si nécessaire, ici 0)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)

losses = []

In [ ]:
epochs = 50

print(f"--- Démarrage de l'entraînement sur {epochs} époques ---")

for epoch in range(epochs):
    optimizer.zero_grad() # Remise à zéro des gradients
    
    # 1. Forward Pass
    # Note: En entraînement, on utilise un masque look-ahead pour empêcher de tricher
    mask = create_look_ahead_mask(decoder_input_train.size(1)).to(decoder_input_train.device)
    
    # L'encodeur traite la source
    enc_output = model.encode(src_seq)
    
    # Le décodeur traite l'input cible (Teacher Forcing : on lui donne la vraie réponse précédente)
    # output shape: (Batch, Seq_Len, d_model)
    dec_output, _ = model.decode(decoder_input_train, enc_output, mask)
    
    # Projection vers le vocabulaire (Batch, Seq_Len, Vocab_Size)
    logits = model.final_linear(dec_output)
    
    # 2. Calcul de la Loss
    # On doit aplatir les tenseurs pour CrossEntropy : (Batch * Seq_Len, Vocab_Size) vs (Batch * Seq_Len)
    loss = criterion(logits.view(-1, tokenizer.vocab_size), decoder_targets_train.reshape(-1))
    
    # 3. Backward Pass (Calcul des gradients)
    loss.backward()
    
    # 4. Mise à jour des poids
    optimizer.step()
    
    # Suivi
    losses.append(loss.item())
    
    if (epoch + 1) % 10 == 0:
        print(f"Époque {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

print("Entraînement terminé.")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(losses, label="Training Loss")
plt.xlabel("Époques")
plt.ylabel("Cross Entropy Loss")
plt.title("Convergence du Modèle (Overfitting sur 1 phrase)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
model.eval() # Mode évaluation

# Départ avec juste le token [CLS]
test_decoder_input = torch.tensor([[tokenizer.cls_token_id]])
max_len = src_seq.size(1) + 2 # Longueur de la phrase originale + marge

print("\n--- Test après entraînement ---")
with torch.no_grad():
    enc_output = model.encode(src_seq) # La source est toujours la même
    
    for i in range(max_len):
        mask = create_look_ahead_mask(test_decoder_input.size(1))
        dec_output, weights = model.decode(test_decoder_input, enc_output, mask)
        
        # Prédiction
        logits = model.final_linear(dec_output[:, -1, :])
        predicted_id = torch.argmax(logits, dim=-1)
        
        # Arrêt si on prédit [SEP] (Token de fin)
        if predicted_id.item() == tokenizer.sep_token_id:
            print("Token [SEP] prédit -> Fin de génération.")
            break
            
        test_decoder_input = torch.cat([test_decoder_input, predicted_id.unsqueeze(0)], dim=1)
        print(f"Généré : {tokenizer.decode(predicted_id)}")

full_sentence = tokenizer.decode(test_decoder_input[0], skip_special_tokens=True)
print(f"\nRésultat Final : '{full_sentence}'")

Cellule 12 : Multi-Head Attention avec KV CacheMathématiques du Cache (Slide 29) :À l'étape $t$, nous avons seulement besoin du vecteur Query du token actuel $q_t$.Cependant, pour l'attention, nous avons besoin de tous les Keys et Values précédents.Au lieu de recalculer $K_{0...t}$ et $V_{0...t}$ :On stocke $K_{prev}$ et $V_{prev}$ en mémoire.On calcule uniquement $k_t$ et $v_t$.On concatène : $K_{new} = [K_{prev}, k_t]$.

In [ ]:
class CachedMultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_k = d_model // n_head
        self.d_model = d_model
        
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None, cached_kv=None):
        # q, k, v shape: (Batch, Seq_Len, d_model)
        bs = q.size(0)
        
        # 1. Projections (Pour k et v, on ne projette que les NOUVEAUX tokens si cache existe)
        w_q = self.w_q(q).view(bs, -1, self.n_head, self.d_k).transpose(1, 2) # (B, H, 1, Dk) en génération
        w_k = self.w_k(k).view(bs, -1, self.n_head, self.d_k).transpose(1, 2)
        w_v = self.w_v(v).view(bs, -1, self.n_head, self.d_k).transpose(1, 2)
        
        # 2. Gestion du Cache KV
        if cached_kv is not None:
            past_k, past_v = cached_kv
            # Concaténation sur la dimension Sequence (dim=2)
            # Math: K_total = Concat(K_cache, K_new)
            w_k = torch.cat([past_k, w_k], dim=2)
            w_v = torch.cat([past_v, w_v], dim=2)
            
        # On sauvegarde le nouvel état pour le retourner
        current_cache = (w_k, w_v)
        
        # 3. Attention (Scaled Dot-Product)
        # Attention: w_q a une longueur de 1, w_k a une longueur de T
        scores = torch.matmul(w_q, w_k.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            # Le masque doit correspondre à la forme (1, 1, 1, Total_Len)
            scores = scores.masked_fill(mask == 0, -1e9)
        
        p_attn = F.softmax(scores, dim=-1)
        out = torch.matmul(p_attn, w_v)
        
        # 4. Recombination
        out = out.transpose(1, 2).contiguous().view(bs, -1, self.d_model)
        return self.fc_out(out), current_cache

# --- Mise à jour de la couche Décodeur pour accepter le cache ---
class CachedDecoderLayer(nn.Module):
    def __init__(self, d_model, n_head, d_ff):
        super().__init__()
        self.self_mha = CachedMultiHeadAttention(d_model, n_head)  # Version avec Cache
        self.cross_mha = MultiHeadAttention(d_model, n_head)       # Standard (Cross ne change pas)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, enc_output, padding_mask, self_attn_cache=None):
        # A. Self-Attention (Avec Cache)
        # Si cache existe, x est seulement le dernier token généré
        attn1, new_cache = self.self_mha(x, x, x, mask=None, cached_kv=self_attn_cache)
        x = self.norm1(x + attn1)
        
        # B. Cross-Attention
        # Note: Dans une implémentation complète optimisée, on pourrait aussi cacher 
        # les clés/valeurs de l'encodeur car enc_output est statique.
        attn2, _ = self.cross_mha(q=x, k=enc_output, v=enc_output, mask=padding_mask)
        x = self.norm2(x + attn2)
        
        # C. FFN
        x = self.norm3(x + self.ffn(x))
        
        return x, new_cache

In [ ]:
class CachedTransformer(nn.Module):
    def __init__(self, d_model, n_head, d_ff, tokenizer):
        super().__init__()
        self.d_model = d_model
        self.tokenizer = tokenizer
        
        self.encoder_embedding = nn.Embedding(tokenizer.vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tokenizer.vocab_size, d_model)
        self.enc_pe = PositionalEncoding(d_model)
        self.dec_pe = PositionalEncoding(d_model)
        
        # Pour simplifier, 1 seule couche ici, mais fonctionne avec N
        self.encoder = EncoderLayer(d_model, n_head, d_ff)
        self.decoder = CachedDecoderLayer(d_model, n_head, d_ff)
        self.final_linear = nn.Linear(d_model, tokenizer.vocab_size)

    def encode(self, src_idx):
        x = self.encoder_embedding(src_idx)
        x = self.enc_pe(x)
        return self.encoder(x)

    def generate_step(self, x_input, enc_output, cache=None):
        # 1. Embedding + PE
        # Attention: Si on utilise le cache, x_input est juste le dernier token (pos T)
        # Il faut ajuster le Positional Encoding pour qu'il sache qu'il est à la pos T
        # (Pour simplifier ici, on applique le PE standard, mais en rigueur il faut un offset)
        
        x = self.decoder_embedding(x_input)
        
        # Astuce simple pour PE avec cache:
        # Si cache est None, pos = 0..L. Si cache existe, pos = L+1
        step_offset = 0 if cache is None else cache[0].size(2)
        # On crée un PE à la volée pour ce seul token à la position correcte (simplification)
        current_pe = self.dec_pe.pe[:, step_offset:step_offset+x.size(1)]
        x = x + current_pe
        
        # 2. Passage Decodeur
        x, new_cache = self.decoder(x, enc_output, padding_mask=None, self_attn_cache=cache)
        
        # 3. Logits
        logits = self.final_linear(x)
        return logits, new_cache

Cellule 14 : Benchmarking - Naïf vs KV Cache
Voici la démonstration concrète de la différence.

In [ ]:
import time

# --- Setup ---
d_model = 128
model = CachedTransformer(d_model, n_head=4, d_ff=512, tokenizer=tokenizer)
model.eval()

# Données : "Les transformeurs sont" -> on veut générer la suite
src_sentence = "Les transformeurs sont"
src_tokens = torch.tensor([tokenizer.encode(src_sentence)])
start_token = torch.tensor([[tokenizer.word2idx["<start>"]]])

# Pré-encodage (commun aux deux méthodes)
with torch.no_grad():
    enc_output = model.encode(src_tokens)

# ==========================================
# MÉTHODE 1 : GÉNÉRATION NAÏVE (Sans Cache)
# ==========================================
print("--- Génération Naïve ---")
curr_seq = start_token.clone()
t0 = time.time()

for _ in range(10):
    # On repasse TOUTE la séquence à chaque fois
    # Note: Dans notre classe CachedTransformer, si on passe cache=None, ça agit comme du standard
    logits, _ = model.generate_step(curr_seq, enc_output, cache=None)
    next_token = torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(0)
    curr_seq = torch.cat([curr_seq, next_token], dim=1)

print(f"Temps Naïf: {time.time() - t0:.5f} sec")
print(f"Sortie: {tokenizer.decode(curr_seq[0].numpy())}")


# ==========================================
# MÉTHODE 2 : GÉNÉRATION AVEC KV CACHE
# ==========================================
print("\n--- Génération KV Cache ---")
curr_token = start_token.clone()
cache = None # Initialisation vide
gen_seq = [start_token.item()]
t0 = time.time()

for _ in range(10):
    # On passe SEULEMENT le dernier token et le cache
    logits, cache = model.generate_step(curr_token, enc_output, cache=cache)
    
    # Prédiction
    next_token_val = torch.argmax(logits[:, -1, :], dim=-1)
    gen_seq.append(next_token_val.item())
    
    # Mise à jour pour la prochaine boucle : l'entrée est JUSTE le nouveau token
    curr_token = next_token_val.unsqueeze(0)

print(f"Temps Cache: {time.time() - t0:.5f} sec")
print(f"Sortie: {tokenizer.decode(gen_seq)}")

# NOTE : Sur de très petites séquences et modèles (comme ici), la gestion du cache Python 
# peut parfois être aussi lente que le calcul matriciel brut. 
# Le gain devient massif quand seq_len > 500 ou d_model est grand.